# Quick Test — Your Handwriting via One-DM (Colab GPU, NO training)

The fastest honest way to see your own handwriting synthesized:
pretrained One-DM + your one photo. Nothing here trains anything.

Steps: GPU check → clone official repo → pretrained checkpoint →
upload your photo → preprocess → generate → TrOCR read-back judge → compare.

In [ ]:
!nvidia-smi  # Runtime -> Change runtime type -> T4 GPU

In [ ]:
# Clone official One-DM (MIT) + deps
!git clone --depth 1 https://github.com/dailenson/One-DM /content/One-DM
%cd /content/One-DM
!pip install -q -r requirements.txt gdown opencv-python

## Pretrained checkpoint
Copy the Google Drive file id from the One-DM README (posted 2024-10-24).
**Never train from scratch** — this checkpoint is the whole point.

In [ ]:
GDRIVE_ID = 'PASTE_FROM_ONEDM_README'  # <-- fill this
!gdown $GDRIVE_ID -O one_dm_pretrained.pth
# SD 1.5 VAE downloads automatically on first run (from Hugging Face)

## Upload ONE photo of your handwriting
One clean line on plain white paper, good light, no shadows.

In [ ]:
from google.colab import files
uploaded = files.upload()
SAMPLE = list(uploaded.keys())[0]
print('style sample:', SAMPLE)

## Preprocess: deskew → binarize → crop → 64px height

In [ ]:
import cv2, numpy as np

img = cv2.imread(SAMPLE, cv2.IMREAD_GRAYSCALE)
bw = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
coords = np.column_stack(np.where(bw > 0))
angle = cv2.minAreaRect(coords)[-1]
angle = -(90 + angle) if angle < -45 else -angle
h, w = bw.shape
M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
bw = cv2.warpAffine(bw, M, (w, h), borderValue=0)
ys, xs = np.where(bw > 0)
crop = bw[ys.min():ys.max()+1, xs.min():xs.max()+1]
crop = cv2.resize(crop, None, fx=64/crop.shape[0], fy=64/crop.shape[0],
                  interpolation=cv2.INTER_AREA)
cv2.imwrite('style_ref.png', 255 - crop)
print('style_ref.png ready')

## Generate (edit TARGET_TEXT)

In [ ]:
TARGET_TEXT = 'The quick brown fox jumps over the lazy dog'

!python test.py \
    --cfg configs/IAM64.yml \
    --one_dm one_dm_pretrained.pth \
    --generate_type iv_u \
    --device cuda \
    --sampling_timesteps 50 \
    --sample_method ddim \
    --dir /content/outputs

import glob
print('generated:', glob.glob('/content/outputs/**/*.png', recursive=True)[:5])

## TrOCR read-back judge (auto quality gate)
If CER > 0.15 the line is illegible — re-roll with a different seed/steps.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
judge = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')

def read_back(path):
    px = processor(Image.open(path).convert('RGB'), return_tensors='pt').pixel_values
    return processor.batch_decode(judge.generate(px), skip_special_tokens=True)[0]

def cer(a, b):
    a, b = a.lower().strip(), b.lower().strip()
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1] / max(1, len(a))

gen = sorted(glob.glob('/content/outputs/**/*.png', recursive=True))[-1]
recognized = read_back(gen)
score = cer(TARGET_TEXT, recognized)
print(f'expected  : {TARGET_TEXT}')
print(f'recognized: {recognized}')
print(f'CER {score:.3f} ->', 'PASS' if score <= 0.15 else 'FAIL, re-roll')

## Compare: your original vs generated

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(12, 4))
axes[0].imshow(Image.open(SAMPLE)); axes[0].set_title('Your sample')
axes[1].imshow(Image.open(gen)); axes[1].set_title('Generated')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()